# 8.2. Networks Using Blocks \(VGG\)

AlexNet demonstrated it's feasible to learn convolution kernels through convolutional neural networks \(CNN\) and that CNNs generally provide much better accuracy compared to traditional methods at scale. However, each layer of AlexNet was still manually specified in a sense and it did not provide radically new ideas or architectures that accelerated the field of computer vision compared to LeNet from the early 2000s.

In 2014, the Visual Geometry Group \(VGG\) from Oxford University proposed that instead of constructing individual _layers_ of CNNs, we can think of CNNs as being composed by _blocks_. Each block contains 1 or more convolutions followed by activation in succession, ending with a max pooling layer for downsampling. This block-based approach proved useful and enabled practitioners to think of entire _families_ of neural networks instead of just individual networks, allowing them to iterate rapidly by operating at a higher level of abstraction.

Software used in this notebook based on [AtomGit AI](https://ai.gitcode.com/docs/notebooks/free-usage/) free notebook environment.

1. Python 3.11
1. MindSpore 2.8.0
1. CANN 8.5.0

In [1]:
%pip install mindspore==2.8.0 \
    -i https://repo.mindspore.cn/pypi/simple \
    --trusted-host repo.mindspore.cn \
    --extra-index-url https://repo.huaweicloud.com/repository/pypi/simple

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://repo.mindspore.cn/pypi/simple, https://repo.huaweicloud.com/repository/pypi/simple

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


MindSpore version:  2.8.0


[WARNING] DEVICE(16407,ffff1cb4f120,python3.11):2026-05-09-19:00:05.135.680 [mindspore/ccsrc/plugin/ascend/res_manager/mem_manager/ascend_vmm_adapter.h:176] CheckVmmDriverVersion] Open file /etc/ascend_install.info failed.
[WARNING] DEVICE(16407,ffff1cb4f120,python3.11):2026-05-09-19:00:05.135.740 [mindspore/ccsrc/plugin/ascend/res_manager/mem_manager/ascend_vmm_adapter.h:204] CheckVmmDriverVersion] Open file /usr/local/Ascend/driver/version.info failed.


The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 8.2.1. VGG Blocks

As explained above, a VGG block consists of:

1. $n$ convolutional layers with kernel size of 3 and padding of 1 to preserve the image size while increasing the number of channels for feature extraction. Each convolutional layer is followed by an activation function such as ReLU activation
1. A max pooling layer with kernel size and stride 2 for downsampling

Let's implement it in our function `vgg_block` below. We'll also introduce [batch normalization](https://en.wikipedia.org/wiki/Batch_normalization) after each convolutional layer with activation to prevent vanishing and exploding gradients due to FP16 precision issues.

In [3]:
import mindspore.nn as nn

def vgg_block(num_convs, in_channels, out_channels):
    """num_convs must be a positive integer"""
    layers = []
    layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3))
    layers.append(nn.ReLU())
    layers.append(nn.BatchNorm2d(out_channels))
    for _ in range(num_convs - 1):
        layers.append(nn.Conv2d(out_channels, out_channels, kernel_size=3))
        layers.append(nn.ReLU())
        layers.append(nn.BatchNorm2d(out_channels))
    layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
    return nn.SequentialCell(layers)

vgg_2 = vgg_block(num_convs=2, in_channels=128, out_channels=256)
vgg_2

SequentialCell(
  (0): Conv2d(input_channels=128, output_channels=256, kernel_size=(3, 3), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xffff24c30090>, bias_init=None, format=NCHW)
  (1): ReLU()
  (2): BatchNorm2d(num_features=256, eps=1e-05, momentum=0.9, gamma=Parameter (name=2.gamma, shape=(256,), dtype=Float32, requires_grad=True), beta=Parameter (name=2.beta, shape=(256,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=2.moving_mean, shape=(256,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=2.moving_variance, shape=(256,), dtype=Float32, requires_grad=False))
  (3): Conv2d(input_channels=256, output_channels=256, kernel_size=(3, 3), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xffff2b719d90>, bias_init=None, format=NCHW)
  (4): ReLU()
  (5):

## 8.2.2. VGG Network

Here's the classic VGG-11 network designed for ImageNet. Recall that image samples from ImageNet are colored RGB images $224 \times 224$ pixels in size.

1. 1st VGG block with 1 convolution, 3 input channels, 64 output channels
1. 2nd VGG block with 1 convolution, 64 input channels, 128 output channels
1. 3rd VGG block with 2 convolutions, 128 input channels, 256 output channels
1. 4th VGG block with 2 convolutions, 256 input channels, 512 output channels
1. 5th VGG block with 2 convolutions, 512 input channels, 512 output channels
1. Flattening layer to transform the $7 \times 7 \times 512$ feature maps to 25088 channels
1. 1st fully connected \(FC\) layer with 25088 input channels, 4096 output channels, ReLU activation
1. 1st dropout layer with $p = 0.5$
1. 2nd FC layer with 4096 input channels, 4096 output channels, ReLU activation
1. 2nd dropout layer with $p = 0.5$
1. Final FC layer with 4096 input channels, 1000 output channels

Since this network has 8 convolutional layers and 3 FC layers for a total of 11 convolutional + FC layers, hence its name VGG-11.

Let's adapt it slightly for our Fashion MNIST dataset use case. The only differences are:

1. The 1st VGG block accepts 1 input channel \(grayscale\) instead of 3 input channels \(RGB\)
1. The final FC layer outputs 10 raw logits instead of 1000. Each raw logit corresponds to a class label in our Fashion MNIST dataset
1. Our `vgg_block` function adds a batch normalization layer after each convolution layer with activation

In [4]:
vgg_11 = nn.SequentialCell([
    vgg_block(num_convs=1, in_channels=1, out_channels=64),
    vgg_block(num_convs=1, in_channels=64, out_channels=128),
    vgg_block(num_convs=2, in_channels=128, out_channels=256),
    vgg_block(num_convs=2, in_channels=256, out_channels=512),
    vgg_block(num_convs=2, in_channels=512, out_channels=512),
    nn.Flatten(),
    nn.Dense(25088, 4096, activation='relu'),
    nn.Dropout(p=0.5),
    nn.Dense(4096, 4096, activation='relu'),
    nn.Dropout(p=0.5),
    nn.Dense(4096, 10)
])
vgg_11

SequentialCell(
  (0): SequentialCell(
    (0): Conv2d(input_channels=1, output_channels=64, kernel_size=(3, 3), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xffff25085a50>, bias_init=None, format=NCHW)
    (1): ReLU()
    (2): BatchNorm2d(num_features=64, eps=1e-05, momentum=0.9, gamma=Parameter (name=0.2.gamma, shape=(64,), dtype=Float32, requires_grad=True), beta=Parameter (name=0.2.beta, shape=(64,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=0.2.moving_mean, shape=(64,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=0.2.moving_variance, shape=(64,), dtype=Float32, requires_grad=False))
    (3): MaxPool2d(kernel_size=2, stride=2, pad_mode=VALID)
  )
  (1): SequentialCell(
    (0): Conv2d(input_channels=64, output_channels=128, kernel_size=(3, 3), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init

As usual, let's wrap it with MindSpore AMP to handle type casting between FP32 and FP16 automatically.

In [5]:
import mindspore.amp as amp

vgg_11_amp = amp.auto_mixed_precision(network=vgg_11, amp_level='O2')
vgg_11_amp

_OutputTo32(
  (_backbone): SequentialCell(
    (0): SequentialCell(
      (0): Conv2d(input_channels=1, output_channels=64, kernel_size=(3, 3), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xffff25085a50>, bias_init=None, format=NCHW)
      (1): ReLU()
      (2): _OutputTo16(
        (_backbone): BatchNorm2d(num_features=64, eps=1e-05, momentum=0.9, gamma=Parameter (name=0.2.gamma, shape=(64,), dtype=Float32, requires_grad=True), beta=Parameter (name=0.2.beta, shape=(64,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=0.2.moving_mean, shape=(64,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=0.2.moving_variance, shape=(64,), dtype=Float32, requires_grad=False))
      )
      (3): MaxPool2d(kernel_size=2, stride=2, pad_mode=VALID)
    )
    (1): SequentialCell(
      (0): Conv2d(input_channels=64, output_channels=128, kernel_size=(3, 3), stride=(

Recall our utility function `layer_summary`. Let's use it to inspect the output shape of each VGG block and FC layer starting from a grayscale $224 \times 224$ image.

In [6]:
import mindspore.ops as ops

def layer_summary(net, X_shape):
    print(f'Input shape: {X_shape}')
    X = ops.randn(*X_shape)
    for cell in net.cells():
        X = cell(X)
        print(f'Output shape from {cell.__class__.__name__}: {X.shape}')

X_shape = (1, 1, 224, 224)
layer_summary(net=vgg_11, X_shape=X_shape)

Input shape: (1, 1, 224, 224)
Output shape from SequentialCell: (1, 64, 112, 112)
Output shape from SequentialCell: (1, 128, 56, 56)
Output shape from SequentialCell: (1, 256, 28, 28)
Output shape from SequentialCell: (1, 512, 14, 14)
Output shape from SequentialCell: (1, 512, 7, 7)
Output shape from Flatten: (1, 25088)
Output shape from Dense: (1, 4096)
Output shape from Dropout: (1, 4096)
Output shape from Dense: (1, 4096)
Output shape from Dropout: (1, 4096)
Output shape from Dense: (1, 10)


## 8.2.3. Training

Time to train our VGG-11 network on the Fashion MNIST dataset. Again, we'll use upsampling to ensure each image is $224 \times 224$ pixels.

In [7]:
import os

dataset_dir = 'data/fashion/'
os.makedirs(dataset_dir, exist_ok=True)

In [8]:
import gzip
import urllib.request

prefix_url = 'https://donaldsebleung.com/assets/datasets/fashion-mnist'
X_train_url = f'{prefix_url}/train-images-idx3-ubyte.gz'
y_train_url = f'{prefix_url}/train-labels-idx1-ubyte.gz'
X_test_url = f'{prefix_url}/t10k-images-idx3-ubyte.gz'
y_test_url = f'{prefix_url}/t10k-labels-idx1-ubyte.gz'

X_train_path = os.path.join(dataset_dir, 'train-images-idx3-ubyte')
y_train_path = os.path.join(dataset_dir, 'train-labels-idx1-ubyte')
X_test_path = os.path.join(dataset_dir, 't10k-images-idx3-ubyte')
y_test_path = os.path.join(dataset_dir, 't10k-labels-idx1-ubyte')

with urllib.request.urlopen(X_train_url) as response:
    with open(X_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_train_url) as response:
    with open(y_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(X_test_url) as response:
    with open(X_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_test_url) as response:
    with open(y_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

In [9]:
import mindspore.dataset as ds

train_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='train', shuffle=True)
test_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='test', shuffle=True)

In [10]:
import mindspore.dataset.vision as vision
import mindspore.dataset.transforms as transforms
from mindspore import dtype as mstype

def transform_ds(dataset):
    image_transforms = [
        vision.Resize(size=(224, 224)),
        vision.Rescale(rescale=1/255, shift=0),
        vision.HWC2CHW()
    ]
    label_transforms = [
        transforms.OneHot(num_classes=10),
        transforms.TypeCast(data_type=mstype.float32)
    ]
    dataset = dataset.map(operations=image_transforms, input_columns='image')
    dataset = dataset.map(operations=label_transforms, input_columns='label')
    dataset = dataset.batch(batch_size=128, drop_remainder=False)
    return dataset

train_ds = transform_ds(dataset=train_ds)
test_ds = transform_ds(dataset=test_ds)

In [11]:
loss_fn = nn.SoftmaxCrossEntropyWithLogits(reduction='mean')
loss_fn

SoftmaxCrossEntropyWithLogits()

In [12]:
optimizer = nn.SGD(params=vgg_11_amp.trainable_params(), learning_rate=0.01)
optimizer

SGD()

In [13]:
from mindspore.train import Model

model = Model(network=vgg_11_amp, loss_fn=loss_fn, optimizer=optimizer, metrics={'accuracy', 'loss'})
model

In [14]:
from mindspore.train import EarlyStopping

early_stopping = EarlyStopping(patience=5, verbose=True, restore_best_weights=True)
early_stopping

In [15]:
epochs = 100

In [16]:
model.fit(epoch=epochs,
          train_dataset=train_ds,
          valid_dataset=test_ds,
          callbacks=[early_stopping],
          dataset_sink_mode=True)

path string is NULLpath string is NULL...Restoring model weights from the end of the best epoch.
Epoch 00008: early stopping


Let's check the validation loss and accuracy of our trained VGG-11 model against the Fashion MNIST dataset.

In [17]:
metrics = model.eval(valid_dataset=test_ds)
val_acc = metrics['accuracy']
val_loss = metrics['loss']
print(f'Validation loss: {val_loss:.4f}')
print(f'Validation accuracy: {val_acc:.4f}')

Validation loss: 0.2315
Validation accuracy: 0.9149


The validation accuracy of our improvised variation of VGG-11 with batch normalization reaches $90\%$ accuracy easily - a noticeable improvement over AlexNet and definitely much better than the pure FC models we encountered in earlier chapters!

## 8.2.4. Summary

In this chapter, we saw how VGG introduced the notion of convolutional _blocks_ which allows us to easily define an entire _family_ of neural networks and accelerate the practice of deep learning. In some sense, VGG is the first truly modern deep network.